In [ ]:
import zipfile
import os
import base64
import io
import time
import requests
import chardet
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display
from PIL import Image as im
import seaborn as sns

In [ ]:
def mm(graph, retries=5):
    graphbytes = graph.encode("utf8")
    base64_bytes = base64.urlsafe_b64encode(graphbytes)
    base64_string = base64_bytes.decode("ascii")
    url = 'https://mermaid.ink/img/' + base64_string

    for attempt in range(retries):
        try:
            response = requests.get(url, timeout=10)
            response.raise_for_status()
            img = im.open(io.BytesIO(response.content))
            plt.figure(figsize=(14, max(8, len(graph.splitlines()) * 0.4)))
            plt.imshow(img)
            plt.axis('off')
            plt.show()
            return
        except Exception as e:
            if attempt < retries - 1:
                wait = 2 ** attempt
                print(f"Attempt {attempt + 1} failed: {e}. Retrying in {wait}s...")
                time.sleep(wait)
            else:
                print(f"All {retries} attempts failed: {e}")

In [ ]:
def extract_zip(zip_filename, output_dir):
    """
    Extract a ZIP file to a specified directory.
    
    Parameters:
    zip_filename (str): Path to the ZIP file.
    output_dir (str): Directory where to extract the contents.
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
        zip_ref.extractall(output_dir)
    
    print(f"Extracted {zip_filename} to {output_dir}")

In [ ]:
pwd = os.getcwd()
print("Current working directory:", pwd)

input_file = os.path.join(pwd, 'input', 'archive.zip')
output_dir = os.path.join(pwd)

extract_zip(input_file, output_dir)

In [ ]:
def display_info(df):
    """
    Display basic information about the DataFrame.
    """
    print(f"Shape: {df.shape}")
    print(f"Number of rows: {df.shape[0]}")
    print(f"Number of columns: {df.shape[1]}")

    print()
    print("--- Data Types ---")
    print(df.dtypes)

    print()
    print("--- Memory Usage ---")
    print(df.memory_usage(deep=True))

    print()
    print("--- Unique Values ---")
    print(df.nunique())

    print()
    print("--- Numeric Summary ---")
    print(df.describe())

    print()
    print("--- Categorical Summary ---")
    cat_cols = df.select_dtypes(include='object').columns
    if len(cat_cols) > 0:
        print(df[cat_cols].describe())
    else:
        print("No categorical columns")

In [ ]:
def display_column_datatype(df, column_names):
    """
    Display the data type of specified columns.
    """
    if not column_names:
        print("No columns provided. Nothing to do.")
        return

    print("--- Column Data Types ---")
    for col in column_names:
        if col not in df.columns:
            print(f"  {col}: not found")
        else:
            print(f"  {col}: {df[col].dtype}")

In [ ]:
def mindmap_columns(df):
    """
    Draw a mindmap of DataFrame columns using Mermaid.js, grouped by data type.
    """
    # Group columns by dtype
    dtype_groups = {}
    for col in df.columns:
        dtype_str = str(df[col].dtype)
        if dtype_str not in dtype_groups:
            dtype_groups[dtype_str] = []
        dtype_groups[dtype_str].append(col)

    # Build Mermaid mindmap syntax
    lines = ["mindmap", "  root((DataFrame))"]
    for dtype_str, cols in dtype_groups.items():
        lines.append(f"    {dtype_str}")
        for col in cols:
            lines.append(f"      {col}")

    graph = "\n".join(lines)
    mm(graph)

In [ ]:
def missing_summary(df):
    """
    Display missing values count and percentage per column.
    """
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    missing_df = pd.DataFrame({'Missing': missing, 'Percentage': missing_pct})
    missing_with_values = missing_df[missing_df['Missing'] > 0]

    print("--- Missing Values ---")
    if len(missing_with_values) > 0:
        print(missing_with_values)
    else:
        print("No missing values")
    print()
    print(f"Total missing: {missing.sum()}")

In [ ]:
def duplicate_summary(df):
    """
    Display duplicate row count and show duplicates if any.
    """
    dup_count = df.duplicated().sum()
    print("--- Duplicates ---")
    print(f"Duplicate rows: {dup_count}")
    if dup_count > 0:
        print()
        print("Duplicate rows:")
        print(df[df.duplicated(keep=False)])

In [ ]:
def encoding_summary(file_path):
    """
    Detect and display the character encoding of a file,
    and check string columns for non-ASCII characters.
    """
    with open(file_path, 'rb') as f:
        raw = f.read()

    result = chardet.detect(raw)
    print("--- File Encoding ---")
    print(f"  Encoding:   {result['encoding']}")
    print(f"  Confidence: {result['confidence']:.2%}")
    print(f"  Language:   {result.get('language', 'N/A')}")
    print()

    df = pd.read_csv(file_path)
    str_cols = df.select_dtypes(include='object').columns

    print("--- Non-ASCII Characters in String Columns ---")
    found = False
    for col in str_cols:
        non_ascii = df[col].dropna().apply(lambda x: any(ord(c) > 127 for c in str(x)))
        count = non_ascii.sum()
        if count > 0:
            found = True
            print(f"  {col}: {count} rows with non-ASCII characters")
            samples = df[col][non_ascii].head(3).tolist()
            for s in samples:
                print(f"    Example: {s}")
    if not found:
        print("  All string columns contain only ASCII characters")

In [ ]:
def display_rows(df, n=5):
    """
    Display the first and last n rows of the DataFrame as formatted tables.
    If n != len(df), display both first and last n rows.
    If n == len(df), display all rows in one table.
    """
    def print_table(title, data):
        table = data.to_string()
        lines = table.split('\n')
        width = max(len(line) for line in lines)
        sep = '-' * width
        print(title)
        print(sep)
        print(lines[0])
        print(sep)
        for line in lines[1:]:
            print(line)
        print(sep)

    if n == len(df):
        # Display all rows
        print_table(f"--- All {n} Rows ---", df)
    else:
        # Display both first and last n rows
        print_table(f"--- First {n} Rows ---", df.head(n))
        print()
        print_table(f"--- Last {n} Rows ---", df.tail(n))

In [ ]:
def datetime_formatting(df, column_names):
    """
    Parse specified columns as datetime and display summary info.
    Returns the modified DataFrame.
    """
    if not column_names:
        print("No columns provided. Nothing to do.")
        return df

    df = df.copy()

    for col in column_names:
        if col not in df.columns:
            print(f"Column '{col}' not found in DataFrame. Skipping.")
            continue

        print(f"--- {col} ---")
        print(f"  Before: {df[col].dtype}")

        df[col] = pd.to_datetime(df[col], errors='coerce')

        print(f"  After:  {df[col].dtype}")
        parsed = df[col].dropna()
        if len(parsed) > 0:
            print(f"  Min:    {parsed.min()}")
            print(f"  Max:    {parsed.max()}")
            print(f"  Range:  {parsed.max() - parsed.min()}")
        failed = df[col].isna().sum() - df[col].isna().sum()
        original_nulls = pd.to_datetime(df[col], errors='coerce').isna().sum()
        print(f"  Parsed: {len(parsed)} / {len(df)}")
        print()

    display_rows(df)
    return df

In [ ]:
def normalize_case(df, column_names=None):
    """
    Convert string columns to uppercase.
    Returns the modified DataFrame.
    """
    df = df.copy()

    if column_names is None:
        column_names = df.select_dtypes(include='object').columns.tolist()

    if not column_names:
        print("No string columns to normalize.")
        return df

    print("--- Normalize Case (Uppercase) ---")
    for col in column_names:
        if col not in df.columns:
            print(f"  Column '{col}' not found. Skipping.")
            continue

        df[col] = df[col].str.upper()
        print(f"  {col}: converted to uppercase")

    print()
    display_rows(df)
    return df

In [ ]:
def drop_columns(df, column_names):
    """
    Drop specified columns from the DataFrame.
    Returns the modified DataFrame.
    """
    if not column_names:
        print("No columns provided. Nothing to do.")
        return df

    df = df.copy()

    print("--- Drop Columns ---")
    for col in column_names:
        if col not in df.columns:
            print(f"  {col}: not found. Skipping.")
        else:
            df = df.drop(columns=[col])
            print(f"  {col}: dropped")

    print()
    print(f"Remaining columns: {len(df.columns)}")
    return df

In [ ]:
def whitespace_trimming(df, column_names=None):
    """
    Strip leading and trailing whitespace from string columns.
    Returns the modified DataFrame.
    """
    df = df.copy()

    if column_names is None:
        column_names = df.select_dtypes(include='object').columns.tolist()

    if not column_names:
        print("No string columns to trim.")
        return df

    print("--- Whitespace Trimming ---")
    for col in column_names:
        if col not in df.columns:
            print(f"  Column '{col}' not found. Skipping.")
            continue

        before = df[col].copy()
        df[col] = df[col].str.strip()
        trimmed = (before != df[col]).sum()
        print(f"  {col}: {trimmed} values trimmed")

    print()
    display_rows(df)
    return df

In [ ]:
def column_classification(df, column_name):
    """
    Classify a column and return unique items.
    Replaces column values with their index in the unique items list.
    Returns a tuple of (updated df, list of unique items).
    """
    if column_name not in df.columns:
        print(f"Column '{column_name}' not found.")
        return df, []

    df = df.copy()
    unique_items = df[column_name].dropna().unique().tolist()
    value_to_index = {v: i for i, v in enumerate(unique_items)}

    print(f"--- {column_name} ---")
    print(f"  Dtype:   {df[column_name].dtype}")
    print(f"  Unique:  {len(unique_items)}")
    print(f"  Mapping:")
    for v, i in value_to_index.items():
        print(f"    {v} -> {i}")

    df[column_name] = df[column_name].map(value_to_index)

    return df, unique_items

In [ ]:
def column_uniques(df, column_name, verbose=True):
    """
    Get unique items from a column.
    Returns a list of unique values (excluding NaN).
    """
    if column_name not in df.columns:
        if verbose:
            print(f"Column '{column_name}' not found.")
        return []
    
    unique_items = df[column_name].dropna().unique().tolist()
    
    if verbose:
        print(f"--- {column_name} ---")
        print(f"  Dtype:   {df[column_name].dtype}")
        print(f"  Unique:  {len(unique_items)}")
        print(f"  Items:")
        for i, item in enumerate(unique_items):
            print(f"    {i}: {item}")
    
    return unique_items

In [ ]:
def datetime_age_classification(df, column_name, target_column):
    """
    Calculate age in fractional years from a datetime column.
    Adds a new target_column with the computed age values.
    Returns the modified DataFrame.
    """
    if column_name not in df.columns:
        print(f"Column '{column_name}' not found.")
        return df

    if target_column in df.columns:
        print(f"Column '{target_column}' already exists.")
        return df

    if not pd.api.types.is_datetime64_any_dtype(df[column_name]):
        print(f"Column '{column_name}' is not datetime dtype.")
        return df

    df = df.copy()
    now = pd.Timestamp.now()
    df[target_column] = (now - df[column_name]).dt.days / 365.25

    print(f"--- {target_column} ---")
    print(f"  Source:  {column_name}")
    print(f"  Now:     {now}")
    print(f"  Min:     {df[target_column].min():.2f} years")
    print(f"  Max:     {df[target_column].max():.2f} years")
    print(f"  Mean:    {df[target_column].mean():.2f} years")

    return df

In [ ]:
def range_summary(df):
    """
    Print min, max, mean, std deviation, and other stats for all numeric columns.
    """
    numeric_cols = df.select_dtypes(include='number').columns

    if len(numeric_cols) == 0:
        print("No numeric columns found.")
        return

    rows = []
    for col in numeric_cols:
        mean = df[col].mean()
        std = df[col].std()

        rows.append({
            'Column':   col,
            'Count':    df[col].count(),
            'Nulls':    df[col].isna().sum(),
            'Min':      round(df[col].min(), 4),
            'Max':      round(df[col].max(), 4),
            'Range':    round(df[col].max() - df[col].min(), 4),
            'Mean':     round(mean, 4),
            'Median':   round(df[col].median(), 4),
            'Mode':     round(df[col].mode()[0], 4),
            'Std':      round(std, 4),
            'Variance': round(df[col].var(), 4),
        })

    stats_df = pd.DataFrame(rows).set_index('Column')
    table = stats_df.to_string()
    lines = table.split('\n')
    width = max(len(line) for line in lines)
    sep = '-' * width

    print("--- Numeric Column Ranges ---")
    print(sep)
    print(lines[0])
    print(sep)
    for line in lines[1:]:
        print(line)
    print(sep)

In [ ]:
def plot_spread(values, column_name):
    """
    Plot histogram and boxplot side by side for a given series.
    """
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(values, bins=30, color='salmon', edgecolor='black')
    axes[0].set_title(f'{column_name} — Histogram')
    axes[0].set_xlabel(column_name)
    axes[0].set_ylabel('Frequency')

    axes[1].boxplot(values, vert=True)
    axes[1].set_title(f'{column_name} — Boxplot')
    axes[1].set_ylabel(column_name)

    plt.tight_layout()
    plt.show()

In [ ]:
def extract_column(df, column_name):
    """
    Extract a column's non-null values as a Series.
    """
    if column_name not in df.columns:
        print(f"Column '{column_name}' not found.")
        return pd.Series(dtype='float64')

    return df[column_name].dropna()

In [ ]:
def count_negatives(df, column_name):
    """
    Count negative values in a column and display their spread.
    Returns a tuple of (negative count, total count, probability).
    """
    if column_name not in df.columns:
        print(f"Column '{column_name}' not found.")
        return 0, 0, 0.0

    total = df[column_name].count()
    neg_mask = df[column_name] < 0
    negatives = neg_mask.sum()
    probability = negatives / total if total > 0 else 0.0

    print(f"--- {column_name} ---")
    print(f"  Negatives:   {negatives}")
    print(f"  Total:       {total}")
    print(f"  Probability: {probability:.4f}")

    if negatives > 0:
        neg_values = df.loc[neg_mask, column_name]
        print(f"  Min:         {neg_values.min():.4f}")
        print(f"  Max:         {neg_values.max():.4f}")
        print(f"  Mean:        {neg_values.mean():.4f}")
        print(f"  Median:      {neg_values.median():.4f}")
        print(f"  Std:         {neg_values.std():.4f}")
        plot_spread(neg_values, f'{column_name} (negatives)')

    return negatives, total, probability

In [ ]:
def invert_negatives(df, column_names):
    """
    Invert negative values to positive by multiplying them by -1.
    Returns the modified DataFrame.
    """
    if not column_names:
        print("No columns provided. Nothing to do.")
        return df

    df = df.copy()

    print("--- Invert Negatives ---")
    for col in column_names:
        if col not in df.columns:
            print(f"  {col}: not found. Skipping.")
            continue

        neg_mask = df[col] < 0
        count = neg_mask.sum()
        df.loc[neg_mask, col] = df.loc[neg_mask, col] * -1
        print(f"  {col}: {count} negatives inverted")

    print()
    display_rows(df)
    return df

In [ ]:
def count_classifications(df, column_name, classifications, verbose=True):
    """
    Count occurrences of each classification in a label-encoded column.
    Returns a dict mapping classification name to count.
    """
    if column_name not in df.columns:
        if verbose:
            print(f"Column '{column_name}' not found.")
        return {}

    counts = df[column_name].value_counts().sort_index()
    result = {}

    if verbose:
        print(f"--- {column_name} ---")
    for idx, name in enumerate(classifications):
        count = counts.get(idx, 0)
        result[name] = count
        if verbose:
            print(f"  {name}: {count}")

    if verbose:
        print(f"  Total: {counts.sum()}")
    return result

In [ ]:
def piechart(counts, title):
    """
    Plot a pie chart from a counts dict {label: count}.
    """
    labels = list(counts.keys())
    values = list(counts.values())

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.pie(values, labels=labels, autopct='%1.1f%%', startangle=90)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

In [ ]:
def piechart_comparison(counts_churned, counts_not_churned, title):
    """
    Plot two pie charts side by side comparing churned vs not churned.
    """
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    labels_churned = list(counts_churned.keys())
    values_churned = list(counts_churned.values())
    
    labels_not_churned = list(counts_not_churned.keys())
    values_not_churned = list(counts_not_churned.values())
    
    axes[0].pie(values_churned, labels=labels_churned, autopct='%1.1f%%', startangle=90)
    axes[0].set_title(f'Churned - {title}')
    
    axes[1].pie(values_not_churned, labels=labels_not_churned, autopct='%1.1f%%', startangle=90)
    axes[1].set_title(f'Not Churned - {title}')
    
    plt.tight_layout()
    plt.show()

In [ ]:
def stacked_bar_comparison(counts_churned, counts_not_churned, title):
    """
    Plot a 100% stacked bar chart comparing churn rate per category.
    Each bar represents a category; segments show % churned vs not churned.
    A dashed line marks the overall churn rate across all categories.
    """
    labels = list(counts_churned.keys())
    churned = [counts_churned.get(l, 0) for l in labels]
    not_churned = [counts_not_churned.get(l, 0) for l in labels]

    totals = [c + n for c, n in zip(churned, not_churned)]
    pct_churned = [c / t * 100 if t > 0 else 0 for c, t in zip(churned, totals)]
    pct_not_churned = [n / t * 100 if t > 0 else 0 for n, t in zip(not_churned, totals)]

    total_all = sum(totals)
    overall_churn_rate = sum(churned) / total_all * 100 if total_all > 0 else 0

    x = range(len(labels))
    fig, ax = plt.subplots(figsize=(max(8, len(labels) * 0.7), 5))

    ax.bar(x, pct_not_churned, label='Not Churned', color='steelblue')
    ax.bar(x, pct_churned, bottom=pct_not_churned, label='Churned', color='salmon')

    ax.axhline(
        y=100 - overall_churn_rate,
        color='red', linestyle='--', linewidth=1.2,
        label=f'Overall churn boundary ({overall_churn_rate:.1f}%)'
    )

    ax.set_xticks(list(x))
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_ylabel('Percentage (%)')
    ax.set_ylim(0, 100)
    ax.set_title(f'{title} — Churn Rate by Category (100% Stacked)')
    ax.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
def part_whole_barplot(df, category_col, total_col, subset_col, title,
                       total_label="Total", subset_label="Subset",
                       xlabel="Count", ylabel=None):
    """
    Plot a horizontal part-whole bar chart using Seaborn.
    A pastel bar shows the total and a muted bar overlays the subset.

    Parameters:
    df            : DataFrame with the data to plot.
    category_col  : Column name for the y-axis categories.
    total_col     : Column name for the full-length bars.
    subset_col    : Column name for the overlaid subset bars.
    title         : Chart title.
    total_label   : Legend label for the total bars.
    subset_label  : Legend label for the subset bars.
    xlabel        : X-axis label.
    ylabel        : Y-axis label (defaults to category_col).
    """
    plot_data = df[[category_col, total_col, subset_col]].copy()

    f, ax = plt.subplots(figsize=(10, max(4, len(plot_data) * 1.2)))

    sns.set_color_codes("pastel")
    sns.barplot(x=total_col, y=category_col, data=plot_data,
                label=total_label, color="b", ax=ax)

    sns.set_color_codes("muted")
    sns.barplot(x=subset_col, y=category_col, data=plot_data,
                label=subset_label, color="r", ax=ax)

    ax.legend(ncol=2, loc="upper left", bbox_to_anchor=(1.01, 1), frameon=True)
    ax.set(xlabel=xlabel, ylabel=ylabel or category_col, title=title)
    sns.despine(left=True, bottom=True)

    plt.tight_layout()
    plt.show()

In [ ]:
def grouped_barplot(df, category_col, value_cols, title,
                    xlabel=None, ylabel="Count",
                    palette="dark", alpha=0.6, height=6, aspect=1.5):
    """
    Plot a grouped barplot using Seaborn's catplot.

    Parameters:
    df            : DataFrame with the data to plot.
    category_col  : Column name for the x-axis categories.
    value_cols    : List of column names to group as bars (e.g. ['Total', 'Churned']).
    title         : Chart title.
    xlabel        : X-axis label (defaults to category_col).
    ylabel        : Y-axis label.
    palette       : Seaborn color palette.
    alpha         : Bar transparency.
    height        : Figure height.
    aspect        : Figure aspect ratio.
    """
    sns.set_theme(style="whitegrid")

    plot_df = df[[category_col] + value_cols].melt(
        id_vars=category_col, var_name='Category', value_name=ylabel
    )

    g = sns.catplot(
        data=plot_df, kind="bar",
        x=category_col, y=ylabel, hue="Category",
        palette=palette, alpha=alpha, height=height, aspect=aspect
    )
    g.despine(left=True)
    g.set_axis_labels(xlabel or category_col, ylabel)
    g.legend.set_title("")
    g.fig.suptitle(title, y=1.02)

    plt.show()

In [ ]:
def filter_dataframe(df, query):
    """
    Filter a DataFrame using a pandas query string.
    Returns the filtered sub-DataFrame.
    """
    result = df.query(query)

    # print(f"--- Filter: {query} ---")
    # print(f"  Before: {len(df)} rows")
    # print(f"  After:  {len(result)} rows")
    # print(f"  Dropped: {len(df) - len(result)} rows")

    return result

In [ ]:
def horizontal_tabs(tabs_dict):
    """
    Display content in horizontal tabs.
    Takes a dict of {tab_name: callable} where each callable produces output.
    """
    tab = widgets.Tab()
    children = []

    for name, func in tabs_dict.items():
        out = widgets.Output()
        with out:
            func()
        children.append(out)

    tab.children = children
    for i, name in enumerate(tabs_dict.keys()):
        tab.set_title(i, name)

    display(tab)

## Exploratory Data Analysis

1. **Load Data** — Read `telecom_churn.csv` into a pandas DataFrame
2. **Display Info** — Shape, dtypes, memory usage, unique counts, numeric and categorical summaries
3. **Mindmap Columns** — Visualize all columns grouped by data type as a Mermaid mindmap
4. **Missing Summary** — Missing values count and percentage per column
5. **Duplicate Summary** — Duplicate row count and details
6. **Encoding Summary** — Detect file encoding and check for non-ASCII characters
7. **Display Rows** — First and last 5 rows of the DataFrame

In [ ]:
df = pd.read_csv(os.path.join(pwd, 'telecom_churn.csv'))
display_info(df)
mindmap_columns(df)
missing_summary(df)
duplicate_summary(df)
encoding_summary(os.path.join(pwd, 'telecom_churn.csv'))
display_rows(df)

## Data Cleanup

1. **Datetime Formatting** — Parse `date_of_registration` to datetime dtype
2. **Encoding Summary** — Detect file encoding and check for non-ASCII characters
3. **Whitespace Trimming** — Strip leading/trailing whitespace from string columns
4. **Normalize Case** — Convert string columns to uppercase
5. **Drop Columns** — Remove `customer_id` and `pincode`
6. **Range Summary** — Min, max, mean, std, and other stats for all numeric columns
7. **Count Negatives** — Check `calls_made`, `sms_sent`, `data_used` for negative values
8. **Invert Negatives** — Multiply negatives by -1 in `calls_made`, `sms_sent`, `data_used` (likely typos)

**Observation:** Inverting negatives to positives — this can be a typo

In [ ]:
display_column_datatype(df, ['date_of_registration'])
df = datetime_formatting(df, ['date_of_registration'])
display_column_datatype(df, ['date_of_registration'])
encoding_summary(os.path.join(pwd, 'telecom_churn.csv'))
df = whitespace_trimming(df)
df = normalize_case(df)
df = drop_columns(df, ['customer_id', 'pincode'])
display_rows(df)
range_summary(df)
count_negatives(df, 'calls_made')
count_negatives(df, 'sms_sent')
count_negatives(df, 'data_used')
# inverting negatives to positives — this can be a typo
df = invert_negatives(df, ['calls_made', 'sms_sent', 'data_used'])

## Data Classification

1. **Column Classification** — Label-encode `gender`, `telecom_partner`, `state`, `city` to numeric indices
2. **Datetime Age Classification** — Compute `registration_age` (years) from `date_of_registration`
3. **Memory Usage Impact** — Compare memory before and after classification

In [ ]:
mem_before = df.memory_usage(deep=True).sum()

df, genders = column_classification(df, 'gender')
df, telecom_partners = column_classification(df, 'telecom_partner')
df, states = column_classification(df, 'state')
df, cities = column_classification(df, 'city')
df = datetime_age_classification(df, 'date_of_registration', 'registration_age')

mem_after = df.memory_usage(deep=True).sum()

print()
print("--- Memory Usage Impact ---")
print(f"  Before: {mem_before:,} bytes ({mem_before / 1024:.2f} KB)")
print(f"  After:  {mem_after:,} bytes ({mem_after / 1024:.2f} KB)")
print(f"  Saved:  {mem_before - mem_after:,} bytes ({(mem_before - mem_after) / 1024:.2f} KB)")
print(f"  Change: {(mem_after - mem_before) / mem_before * 100:.2f}%")
print()

display_rows(df)

## Plot Spreads

1. **Histogram + Boxplot** — Plot distribution spread for `age`, `estimated_salary`, `calls_made`, `sms_sent`, `data_used`

In [ ]:
for col in ['age', 'estimated_salary', 'calls_made', 'sms_sent', 'data_used']:
    plot_spread(extract_column(df, col), col)

## Counting & Value Frequencies

1. **Count Classifications** — Count occurrences of each label-encoded value for `gender`, `telecom_partner`, `state`, `city`
2. **Pie Charts** — Visualize distributions for gender, telecom partner, state, and city

**Observation:** City and state piecharts look ambiguous; there is no coverage of many states in the city.

In [ ]:
gender_counts = count_classifications(df, 'gender', genders)
telecom_partner_counts = count_classifications(df, 'telecom_partner', telecom_partners)
state_counts = count_classifications(df, 'state', states)
city_counts = count_classifications(df, 'city', cities)

horizontal_tabs({
    'Genders': lambda: piechart(gender_counts, 'Gender Distribution'),
    'Partners': lambda: piechart(telecom_partner_counts, 'Telecom Partner Distribution'),
    'States': lambda: piechart(state_counts, 'State Distribution'),
    'Cities': lambda: piechart(city_counts, 'City Distribution'),
})

# city and state piecharts look ambiguous
# there is no coverage of many states in the city

In [ ]:
df_churned = filter_dataframe(df, 'churn == 1')
df_churned_c = filter_dataframe(df, 'churn != 1')
# print(f"Churned customers: {len(df_churned)} rows")
# display_rows(df_churned)
# print(f"Churned customers: {len(df_churned_c)} rows")
# display_rows(df_churned_c)

churn_counts = {
    'Not Churned': (df['churn'] == 0).sum(),
    'Churned': (df['churn'] == 1).sum()
}
piechart(churn_counts, 'Churn Distribution')



## Analysis of `Gender` `State` `City` `Telecom Partner` on Churn

### Key Observations

**No Strong Predictive Power:** The demographic and service attributes (`Gender`, `State`, `City`, `Telecom Partner`) show minimal discriminative power for predicting churn. The probability distributions are nearly identical between churned and not-churned customer groups, indicating these features are not major churn drivers.

**Balanced Distribution Across Groups:** When a category's percentage among churned customers matches its percentage among not-churned customers, it suggests the category is equally represented in both groups regardless of churn status. This signals weak correlation with churn behavior.

**Implications:**
- `Gender` distribution among churned customers closely mirrors the overall customer gender distribution
- `Telecom Partner` categories are proportionally represented in both churned and not-churned groups
- Geographic factors (`State`, `City`) don't show concentrations of churn in specific regions
- **Conclusion:** Churn appears to be driven by factors beyond these demographics - likely service-related metrics (contract type, tenure, monthly charges, service usage patterns) or behavioral indicators rather than customer characteristics

In [ ]:
gender_counts_churned = count_classifications(df_churned, 'gender', genders, verbose=False)
telecom_partner_counts_churned = count_classifications(df_churned, 'telecom_partner', telecom_partners, verbose=False)
state_counts_churned = count_classifications(df_churned, 'state', states, verbose=False)
city_counts_churned = count_classifications(df_churned, 'city', cities, verbose=False)

gender_counts_not_churned = count_classifications(df_churned_c, 'gender', genders, verbose=False)
telecom_partner_counts_not_churned = count_classifications(df_churned_c, 'telecom_partner', telecom_partners, verbose=False)
state_counts_not_churned = count_classifications(df_churned_c, 'state', states, verbose=False)
city_counts_not_churned = count_classifications(df_churned_c, 'city', cities, verbose=False)

# Print churned vs not churned with probabilities
def print_churn_comparison(counts_churned, counts_not_churned, title):
    print(f"\n--- {title} ---")
    print(f"{'Category':<20} {'Churned':<12} {'Not Churned':<15} {'Total':<8} {'Churn %':<10}")
    print("-" * 70)
    
    total_churned = 0
    total_not_churned = 0
    
    for cat in counts_churned.keys():
        churned = counts_churned[cat]
        not_churned = counts_not_churned[cat]
        total = churned + not_churned
        churn_pct = (churned / total * 100) if total > 0 else 0
        
        total_churned += churned
        total_not_churned += not_churned
        
        print(f"{cat:<20} {churned:<12} {not_churned:<15} {total:<8} {churn_pct:<10.2f}%")
    
    total_all = total_churned + total_not_churned
    overall_churn_pct = (total_churned / total_all * 100) if total_all > 0 else 0
    print("-" * 70)
    print(f"{'TOTAL':<20} {total_churned:<12} {total_not_churned:<15} {total_all:<8} {overall_churn_pct:<10.2f}%")

# Print category probabilities within churned and not churned groups
def print_category_probabilities(counts_churned, counts_not_churned, title):
    print(f"\n{'='*70}")
    print(f"PROBABILITY DISTRIBUTION: {title}")
    print(f"{'='*70}")
    
    total_churned = sum(counts_churned.values())
    total_not_churned = sum(counts_not_churned.values())
    
    print(f"\n--- Within CHURNED Customers (n={total_churned}) ---")
    print(f"{'Category':<25} {'Count':<10} {'Probability':<12}")
    print("-" * 50)
    for cat, count in counts_churned.items():
        prob = (count / total_churned * 100) if total_churned > 0 else 0
        print(f"{cat:<25} {count:<10} {prob:<12.2f}%")
    
    print(f"\n--- Within NOT CHURNED Customers (n={total_not_churned}) ---")
    print(f"{'Category':<25} {'Count':<10} {'Probability':<12}")
    print("-" * 50)
    for cat, count in counts_not_churned.items():
        prob = (count / total_not_churned * 100) if total_not_churned > 0 else 0
        print(f"{cat:<25} {count:<10} {prob:<12.2f}%")

print("\n" + "="*70)
print("CHURN ANALYSIS: COUNTS AND PROBABILITIES")
print("="*70)

print_churn_comparison(gender_counts_churned, gender_counts_not_churned, "GENDER")
print_churn_comparison(telecom_partner_counts_churned, telecom_partner_counts_not_churned, "TELECOM PARTNER")
print_churn_comparison(state_counts_churned, state_counts_not_churned, "STATE")
print_churn_comparison(city_counts_churned, city_counts_not_churned, "CITY")

print("\n\n" + "="*70)
print("PROBABILITY DISTRIBUTION WITHIN CHURN GROUPS")
print("="*70)

print_category_probabilities(gender_counts_churned, gender_counts_not_churned, "GENDER")
print_category_probabilities(telecom_partner_counts_churned, telecom_partner_counts_not_churned, "TELECOM PARTNER")
print_category_probabilities(state_counts_churned, state_counts_not_churned, "STATE")
print_category_probabilities(city_counts_churned, city_counts_not_churned, "CITY")

print("\n" + "="*70)
print("CHURN VISUALIZATION: 100% STACKED BAR CHARTS")
print("="*70 + "\n")

horizontal_tabs({
    'Gender': lambda: stacked_bar_comparison(gender_counts_churned, gender_counts_not_churned, 'Gender'),
    'Partner': lambda: stacked_bar_comparison(telecom_partner_counts_churned, telecom_partner_counts_not_churned, 'Telecom Partner'),
    'State': lambda: stacked_bar_comparison(state_counts_churned, state_counts_not_churned, 'State'),
    'City': lambda: stacked_bar_comparison(city_counts_churned, city_counts_not_churned, 'City'),
})

## Affect of Age on Churn

### Key Observation

**Age Shows No Strong Predictive Power for Churn:** The analysis reveals that churn probability is equally distributed across all age groups. There is no significant variation in churn rates by age, indicating that age alone is not a strong discriminator for predicting customer churn. This suggests that churn drivers are independent of customer age and are likely influenced by other factors such as service quality, pricing, contract terms, or usage patterns rather than demographic age characteristics.

In [ ]:
unique_ages = column_uniques(df, 'age', verbose=False)

# Create dataframe for age churn analysis
data = []
for age in sorted(unique_ages):
    df_age = filter_dataframe(df, f'age == {age}')
    df_age_churned = filter_dataframe(df_age, 'churn == 1')
    
    churned_count = (df_age['churn'] == 1).sum()
    not_churned_count = (df_age['churn'] != 1).sum()
    total = churned_count + not_churned_count
    churn_pct = (churned_count / total * 100) if total > 0 else 0
    
    # Calculate percentages within churned customers of this age
    female_pct = (df_age_churned['gender'] == 0).sum() / churned_count * 100 if churned_count > 0 else 0
    male_pct = (df_age_churned['gender'] == 1).sum() / churned_count * 100 if churned_count > 0 else 0
    
    row = {
        'Age': age,
        'Churned': churned_count,
        'Not Churned': not_churned_count,
        'Total': total,
        'Churn %': round(churn_pct, 2),
        'Female %': round(female_pct, 2),
        'Male %': round(male_pct, 2),
    }
    
    # Add individual city percentages
    for i, city in enumerate(cities):
        city_pct = (df_age_churned['city'] == i).sum() / churned_count * 100 if churned_count > 0 else 0
        row[f'{city} %'] = round(city_pct, 2)
    
    # Add individual telecom partner percentages
    for i, partner in enumerate(telecom_partners):
        partner_pct = (df_age_churned['telecom_partner'] == i).sum() / churned_count * 100 if churned_count > 0 else 0
        row[f'{partner} %'] = round(partner_pct, 2)
    
    data.append(row)

age_churn_df = pd.DataFrame(data)
display_rows(age_churn_df, n=len(unique_ages))

## Affect of Age of Registration on Churn

### Key Observation

**Registration Age (Tenure) Shows No Strong Predictive Power for Churn:** Similar to customer age, the analysis reveals that churn probability is equally distributed across all registration age (tenure) groups. There is no significant variation in churn rates based on how long a customer has been registered. This aligns with the age finding and suggests that **temporal factors** (both customer age and account tenure) are not strong discriminators for predicting churn. Instead, churn drivers are likely independent of customer demographics and account history, pointing towards service-related factors such as pricing changes, service quality, contract terms, or usage pattern disruptions as the true churn drivers.

In [ ]:

# Create dataframe for registration age churn analysis
registration_ages = df['registration_age'].dropna().unique()
registration_ages = sorted([round(age, 1) for age in registration_ages])

data = []
for reg_age in registration_ages:
    df_reg_age = df[df['registration_age'].round(1) == reg_age]
    df_reg_age_churned = df_reg_age[df_reg_age['churn'] == 1]
    
    churned_count = (df_reg_age['churn'] == 1).sum()
    not_churned_count = (df_reg_age['churn'] != 1).sum()
    total = churned_count + not_churned_count
    churn_pct = (churned_count / total * 100) if total > 0 else 0
    
    # Calculate percentages within churned customers of this registration age
    female_pct = (df_reg_age_churned['gender'] == 0).sum() / churned_count * 100 if churned_count > 0 else 0
    male_pct = (df_reg_age_churned['gender'] == 1).sum() / churned_count * 100 if churned_count > 0 else 0
    
    row = {
        'Reg Age': reg_age,
        'Churned': churned_count,
        'Not Churned': not_churned_count,
        'Total': total,
        'Churn %': round(churn_pct, 2),
        'Female %': round(female_pct, 2),
        'Male %': round(male_pct, 2),
    }
    
    # Add individual city percentages
    for i, city in enumerate(cities):
        city_pct = (df_reg_age_churned['city'] == i).sum() / churned_count * 100 if churned_count > 0 else 0
        row[f'{city} %'] = round(city_pct, 2)
    
    # Add individual telecom partner percentages
    for i, partner in enumerate(telecom_partners):
        partner_pct = (df_reg_age_churned['telecom_partner'] == i).sum() / churned_count * 100 if churned_count > 0 else 0
        row[f'{partner} %'] = round(partner_pct, 2)
    
    data.append(row)

registration_age_churn_df = pd.DataFrame(data)
display_rows(registration_age_churn_df, n=len(registration_ages))

## Affect of Estimated Salary on Churn

### Methodology

**Salary Binning Approach:** The `estimated_salary` column contains too many unique values to analyze individually, making granular analysis impractical. Instead, we bin salaries into 5 equal-width ranges across the min-max salary span. This aggregation allows us to identify patterns and trends in churn behavior across salary bands while maintaining statistical significance within each group.

**Currency Localization:** Salary ranges are displayed using the Indian Rupee (₹) symbol for contextual relevance. This is purely a display formatting choice and does not affect the underlying statistical observations, churn patterns, or analytical conclusions.

### Key Observation

**Salary (Binned) Shows No Strong Predictive Power for Churn:** Similar to age and registration age, the analysis of binned salary ranges reveals that churn probability is equally distributed across all income brackets. There is no significant variation in churn rates based on salary level, indicating that estimated income alone is not a strong discriminator for predicting customer churn. This finding reinforces the emerging pattern that **demographic and personal financial characteristics are not primary churn drivers**. Instead, churn appears to be driven by service-related factors such as contract terms, service quality, usage patterns, or billing experiences rather than customer income levels.

In [ ]:

unique_salaries = sorted(df['estimated_salary'].dropna().unique().tolist())
print(f"Unique Salaries: {len(unique_salaries)}")
print(f"Min: {min(unique_salaries)}")
print(f"Max: {max(unique_salaries)}")
print(f"Sample: {unique_salaries[:10]}")


# Create salary bins for analysis
min_salary = df['estimated_salary'].min()
max_salary = df['estimated_salary'].max()

# Create 5 equal-width salary bins
salary_bins = np.linspace(min_salary, max_salary, 6)
salary_labels = [f"₹{int(salary_bins[i]):,.0f}-₹{int(salary_bins[i+1]):,.0f}" for i in range(len(salary_bins)-1)]

df['salary_range'] = pd.cut(df['estimated_salary'], bins=salary_bins, labels=salary_labels, include_lowest=True)

# Create dataframe for salary churn analysis
data = []
for salary_range in salary_labels:
    df_salary = df[df['salary_range'] == salary_range]
    df_salary_churned = df_salary[df_salary['churn'] == 1]
    
    churned_count = (df_salary['churn'] == 1).sum()
    not_churned_count = (df_salary['churn'] != 1).sum()
    total = churned_count + not_churned_count
    churn_pct = (churned_count / total * 100) if total > 0 else 0
    
    # Calculate percentages within churned customers of this salary range
    female_pct = (df_salary_churned['gender'] == 0).sum() / churned_count * 100 if churned_count > 0 else 0
    male_pct = (df_salary_churned['gender'] == 1).sum() / churned_count * 100 if churned_count > 0 else 0
    
    row = {
        'Salary Range': salary_range,
        'Churned': churned_count,
        'Not Churned': not_churned_count,
        'Total': total,
        'Churn %': round(churn_pct, 2),
        'Female %': round(female_pct, 2),
        'Male %': round(male_pct, 2),
    }
    
    # Add individual city percentages
    for i, city in enumerate(cities):
        city_pct = (df_salary_churned['city'] == i).sum() / churned_count * 100 if churned_count > 0 else 0
        row[f'{city} %'] = round(city_pct, 2)
    
    # Add individual telecom partner percentages
    for i, partner in enumerate(telecom_partners):
        partner_pct = (df_salary_churned['telecom_partner'] == i).sum() / churned_count * 100 if churned_count > 0 else 0
        row[f'{partner} %'] = round(partner_pct, 2)
    
    data.append(row)

salary_churn_df = pd.DataFrame(data)
display_rows(salary_churn_df, n=len(salary_labels))

## Affect of Number of Dependents on Churn

### Key Observation

**Number of Dependents Shows No Strong Predictive Power for Churn:** Similar to age, registration age, and salary, the analysis reveals that churn probability is equally distributed across all dependent count groups. There is no significant variation in churn rates based on the number of dependents a customer has. This finding continues to reinforce the pattern that **personal and demographic characteristics are not primary churn drivers**. Instead, churn behavior appears to be driven by service-related factors such as contract conditions, service quality, usage patterns, or customer experience rather than family/dependent status.

In [ ]:

# Create dataframe for number of dependents churn analysis
unique_dependents = sorted(df['num_dependents'].dropna().unique().tolist())

data = []
for num_dep in unique_dependents:
    df_dep = df[df['num_dependents'] == num_dep]
    df_dep_churned = df_dep[df_dep['churn'] == 1]
    
    churned_count = (df_dep['churn'] == 1).sum()
    not_churned_count = (df_dep['churn'] != 1).sum()
    total = churned_count + not_churned_count
    churn_pct = (churned_count / total * 100) if total > 0 else 0
    
    # Calculate percentages within churned customers of this dependent count
    female_pct = (df_dep_churned['gender'] == 0).sum() / churned_count * 100 if churned_count > 0 else 0
    male_pct = (df_dep_churned['gender'] == 1).sum() / churned_count * 100 if churned_count > 0 else 0
    
    row = {
        'Dependents': num_dep,
        'Churned': churned_count,
        'Not Churned': not_churned_count,
        'Total': total,
        'Churn %': round(churn_pct, 2),
        'Female %': round(female_pct, 2),
        'Male %': round(male_pct, 2),
    }
    
    # Add individual city percentages
    for i, city in enumerate(cities):
        city_pct = (df_dep_churned['city'] == i).sum() / churned_count * 100 if churned_count > 0 else 0
        row[f'{city} %'] = round(city_pct, 2)
    
    # Add individual telecom partner percentages
    for i, partner in enumerate(telecom_partners):
        partner_pct = (df_dep_churned['telecom_partner'] == i).sum() / churned_count * 100 if churned_count > 0 else 0
        row[f'{partner} %'] = round(partner_pct, 2)
    
    data.append(row)

dependents_churn_df = pd.DataFrame(data)
display_rows(dependents_churn_df, n=len(unique_dependents))

## Affect of Calls Made on Churn

### Methodology

**Calls Made Binning Approach:** The `calls_made` column contains many unique values, making individual analysis impractical. We bin calls into 5 equal-width ranges across the min-max call span. This aggregation allows us to identify patterns in churn behavior across call activity levels while maintaining statistical significance within each group.

### Key Observation

**Calls Made (Binned) Shows No Strong Predictive Power for Churn:** Similar to age, registration age, salary, and number of dependents, the analysis reveals that churn probability is equally distributed across all call volume groups (approximately 19.92%-20.27%). There is no significant variation in churn rates based on the number of calls made, indicating that call activity alone is not a strong discriminator for predicting customer churn. This extends the pattern that **demographic, personal, and behavioral characteristics examined so far are not primary churn drivers**. Churn appears to be driven by other factors such as contract terms, service quality, billing experiences, or customer satisfaction metrics rather than usage frequency.

In [ ]:

unique_calls = sorted(df['calls_made'].dropna().unique().tolist())
print(f"Unique Calls Made: {len(unique_calls)}")
print(f"Min: {min(unique_calls)}")
print(f"Max: {max(unique_calls)}")
print(f"Sample: {unique_calls[:10]}")

# Create calls_made bins for analysis
min_calls = df['calls_made'].min()
max_calls = df['calls_made'].max()

# Create 5 equal-width calls bins
calls_bins = np.linspace(min_calls, max_calls, 6)
calls_labels = [f"{int(calls_bins[i])}-{int(calls_bins[i+1])}" for i in range(len(calls_bins)-1)]

df['calls_range'] = pd.cut(df['calls_made'], bins=calls_bins, labels=calls_labels, include_lowest=True)

# Create dataframe for calls churn analysis
data = []
for calls_range in calls_labels:
    df_calls = df[df['calls_range'] == calls_range]
    df_calls_churned = df_calls[df_calls['churn'] == 1]
    
    churned_count = (df_calls['churn'] == 1).sum()
    not_churned_count = (df_calls['churn'] != 1).sum()
    total = churned_count + not_churned_count
    churn_pct = (churned_count / total * 100) if total > 0 else 0
    
    # Calculate percentages within churned customers of this calls range
    female_pct = (df_calls_churned['gender'] == 0).sum() / churned_count * 100 if churned_count > 0 else 0
    male_pct = (df_calls_churned['gender'] == 1).sum() / churned_count * 100 if churned_count > 0 else 0
    
    row = {
        'Calls Range': calls_range,
        'Churned': churned_count,
        'Not Churned': not_churned_count,
        'Total': total,
        'Churn %': round(churn_pct, 2),
        'Female %': round(female_pct, 2),
        'Male %': round(male_pct, 2),
    }
    
    # Add individual city percentages
    for i, city in enumerate(cities):
        city_pct = (df_calls_churned['city'] == i).sum() / churned_count * 100 if churned_count > 0 else 0
        row[f'{city} %'] = round(city_pct, 2)
    
    # Add individual telecom partner percentages
    for i, partner in enumerate(telecom_partners):
        partner_pct = (df_calls_churned['telecom_partner'] == i).sum() / churned_count * 100 if churned_count > 0 else 0
        row[f'{partner} %'] = round(partner_pct, 2)
    
    data.append(row)

calls_churn_df = pd.DataFrame(data)
display_rows(calls_churn_df, n=len(calls_labels))

In [ ]:

grouped_barplot(
    calls_churn_df,
    category_col="Calls Range",
    value_cols=["Total", "Churned"],
    title="Calls Made vs Churn: Grouped Bar Analysis",
    xlabel="Calls Range",
    ylabel="Number of Customers"
)

## Affect of SMS Sent on Churn

### Methodology

**SMS Sent Binning Approach:** The `sms_sent` column contains many unique values, making individual analysis impractical. We bin SMS counts into 5 equal-width ranges across the min-max SMS span. This aggregation allows us to identify patterns in churn behavior across SMS activity levels while maintaining statistical significance within each group.

### Key Observation

**SMS Sent (Binned) Shows No Strong Predictive Power for Churn:** Similar to age, registration age, salary, number of dependents, and calls made, the analysis reveals that churn probability is equally distributed across all SMS volume groups. There is no significant variation in churn rates based on the number of SMS messages sent, indicating that SMS activity alone is not a strong discriminator for predicting customer churn. This continues the emerging pattern that **multiple service usage and demographic characteristics examined so far are not primary churn drivers**. Churn appears to be driven by other factors beyond activity metrics, such as contract terms, service quality, billing experiences, or customer satisfaction metrics.

In [ ]:

unique_sms = sorted(df['sms_sent'].dropna().unique().tolist())
print(f"Unique SMS Sent: {len(unique_sms)}")
print(f"Min: {min(unique_sms)}")
print(f"Max: {max(unique_sms)}")
print(f"Sample: {unique_sms[:10]}")

# Create sms_sent bins for analysis
min_sms = df['sms_sent'].min()
max_sms = df['sms_sent'].max()

# Create 5 equal-width SMS bins
sms_bins = np.linspace(min_sms, max_sms, 6)
sms_labels = [f"{int(sms_bins[i])}-{int(sms_bins[i+1])}" for i in range(len(sms_bins)-1)]

df['sms_range'] = pd.cut(df['sms_sent'], bins=sms_bins, labels=sms_labels, include_lowest=True)

# Create dataframe for SMS churn analysis
data = []
for sms_range in sms_labels:
    df_sms = df[df['sms_range'] == sms_range]
    df_sms_churned = df_sms[df_sms['churn'] == 1]
    
    churned_count = (df_sms['churn'] == 1).sum()
    not_churned_count = (df_sms['churn'] != 1).sum()
    total = churned_count + not_churned_count
    churn_pct = (churned_count / total * 100) if total > 0 else 0
    
    # Calculate percentages within churned customers of this SMS range
    female_pct = (df_sms_churned['gender'] == 0).sum() / churned_count * 100 if churned_count > 0 else 0
    male_pct = (df_sms_churned['gender'] == 1).sum() / churned_count * 100 if churned_count > 0 else 0
    
    row = {
        'SMS Range': sms_range,
        'Churned': churned_count,
        'Not Churned': not_churned_count,
        'Total': total,
        'Churn %': round(churn_pct, 2),
        'Female %': round(female_pct, 2),
        'Male %': round(male_pct, 2),
    }
    
    # Add individual city percentages
    for i, city in enumerate(cities):
        city_pct = (df_sms_churned['city'] == i).sum() / churned_count * 100 if churned_count > 0 else 0
        row[f'{city} %'] = round(city_pct, 2)
    
    # Add individual telecom partner percentages
    for i, partner in enumerate(telecom_partners):
        partner_pct = (df_sms_churned['telecom_partner'] == i).sum() / churned_count * 100 if churned_count > 0 else 0
        row[f'{partner} %'] = round(partner_pct, 2)
    
    data.append(row)

sms_churn_df = pd.DataFrame(data)
display_rows(sms_churn_df, n=len(sms_labels))

In [ ]:
grouped_barplot(
    sms_churn_df,
    category_col="SMS Range",
    value_cols=["Total", "Churned"],
    title="SMS Sent vs Churn: Grouped Bar Analysis",
    xlabel="SMS Range",
    ylabel="Number of Customers"
)

## Affect of Data Used on Churn

### Methodology

**Data Used Binning Approach:** The `data_used` column contains many unique values, making individual analysis impractical. We bin data usage into 5 equal-width ranges across the min-max data span. This aggregation allows us to identify patterns in churn behavior across data consumption levels while maintaining statistical significance within each group.

### Key Observation

**Data Used (Binned) Shows No Strong Predictive Power for Churn:** Similar to age, registration age, salary, number of dependents, calls made, and SMS sent, the analysis reveals that churn probability is equally distributed across all data usage groups. There is no significant variation in churn rates based on the amount of data used, indicating that data consumption alone is not a strong discriminator for predicting customer churn. This reinforces the comprehensive pattern that **demographic, personal, and service usage characteristics examined are not primary churn drivers**. Churn appears to be driven by other factors beyond activity metrics, such as contract terms, service quality, billing experiences, or customer satisfaction metrics.

In [ ]:

unique_data = sorted(df['data_used'].dropna().unique().tolist())
print(f"Unique Data Used: {len(unique_data)}")
print(f"Min: {min(unique_data)}")
print(f"Max: {max(unique_data)}")
print(f"Sample: {unique_data[:10]}")

# Create data_used bins for analysis
# Use equal-frequency binning (qcut) to ensure balanced bar sizes
df['data_range'], bin_edges = pd.qcut(df['data_used'], q=5, labels=False, retbins=True, duplicates='drop')
data_labels = [f"{int(bin_edges[i])}-{int(bin_edges[i+1])}" for i in range(len(bin_edges)-1)]
df['data_range'] = pd.cut(df['data_used'], bins=bin_edges, labels=data_labels, include_lowest=True)

# Create dataframe for data churn analysis
data = []
for data_range in data_labels:
    df_data = df[df['data_range'] == data_range]
    df_data_churned = df_data[df_data['churn'] == 1]
    
    churned_count = (df_data['churn'] == 1).sum()
    not_churned_count = (df_data['churn'] != 1).sum()
    total = churned_count + not_churned_count
    churn_pct = (churned_count / total * 100) if total > 0 else 0
    
    # Calculate percentages within churned customers of this data range
    female_pct = (df_data_churned['gender'] == 0).sum() / churned_count * 100 if churned_count > 0 else 0
    male_pct = (df_data_churned['gender'] == 1).sum() / churned_count * 100 if churned_count > 0 else 0
    
    row = {
        'Data Range': data_range,
        'Churned': churned_count,
        'Not Churned': not_churned_count,
        'Total': total,
        'Churn %': round(churn_pct, 2),
        'Female %': round(female_pct, 2),
        'Male %': round(male_pct, 2),
    }
    
    # Add individual city percentages
    for i, city in enumerate(cities):
        city_pct = (df_data_churned['city'] == i).sum() / churned_count * 100 if churned_count > 0 else 0
        row[f'{city} %'] = round(city_pct, 2)
    
    # Add individual telecom partner percentages
    for i, partner in enumerate(telecom_partners):
        partner_pct = (df_data_churned['telecom_partner'] == i).sum() / churned_count * 100 if churned_count > 0 else 0
        row[f'{partner} %'] = round(partner_pct, 2)
    
    data.append(row)

data_churn_df = pd.DataFrame(data)
display_rows(data_churn_df, n=len(data_labels))

In [ ]:

# Part-whole bar chart for data_used (horizontal bars)
part_whole_barplot(
    data_churn_df, 
    category_col="Data Range", 
    total_col="Total", 
    subset_col="Churned",
    title="Data Usage vs Churn: Part-Whole Analysis",
    total_label="Total Customers",
    subset_label="Churned Customers",
    xlabel="Number of Customers",
    ylabel="Data Usage Range"
)